In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
train=r"E:\engg\Data\Tumor_data\Train"
test=r"E:\engg\Data\Tumor_data\Test"
val=r"E:\engg\Data\Tumor_data\Val"

In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()])

train_dataset = datasets.ImageFolder(root=train,transform=transform)
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=False,num_workers=2)
mean = torch.zeros(3)
std = torch.zeros(3)
total_images = 0

for images, _ in train_loader:
    batch_size = images.size(0)
    images = images.view(batch_size, 3, -1)
    mean += images.mean(dim=2).sum(dim=0)
    std += images.std(dim=2).sum(dim=0)
    total_images += batch_size
mean /= total_images
std /= total_images

print("Mean:", mean)
print("Std:", std)

Mean: tensor([0.1845, 0.1845, 0.1846])
Std: tensor([0.1796, 0.1796, 0.1796])


In [10]:
#img size= 512,512
#mean,std values calculated from dataset
transform_pipeline = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1845, 0.1845, 0.1846],
    std=[0.1796, 0.1796, 0.1796])
])

In [11]:
train_data= datasets.ImageFolder(train,transform=transform_pipeline)
test_data= datasets.ImageFolder(test,transform=transform_pipeline)
val_data=datasets.ImageFolder(val,transform=transform_pipeline)

train_loader=DataLoader(train_data,batch_size=64,shuffle=True)
test_loader=DataLoader(test_data,batch_size=64,shuffle=False)
val_loader=DataLoader(val_data,batch_size=64,shuffle=False)

In [12]:
print(train_data.classes)

['glioma', 'meningioma', 'notumor', 'pituitary']


In [13]:
class BrainTumour(nn.Module):
    def __init__(self,num_classes=4):
        super().__init__()
        self.features= nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.classifier	= nn.Sequential(
            nn.Flatten(),					
            nn.Linear(128*32*32,256),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(256,	num_classes))    
    def forward(self, x):
        x=self.features(x)
        return	self.classifier(x)

model = BrainTumour(num_classes=4).to(device)

In [14]:
criterion=nn.CrossEntropyLoss()	
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [15]:
epochs=30
best_val_loss = float("inf")
patience = 5
counter = 0

for epoch in range(epochs):
	model.train()
	running_loss=0.0
	for	imgs,labels	in train_loader:
		imgs,labels = imgs.to(device), labels.to(device)
		optimizer.zero_grad()
		outputs= model(imgs)
		loss=criterion(outputs,	labels)	
		loss.backward()	
		optimizer.step()
		running_loss += loss.item()
	train_loss = running_loss / len(train_loader)
	model.eval()
	val_loss = 0.0
	correct = 0
	total = 0      
	with torch.no_grad():
		for imgs, labels in val_loader:
			imgs, labels = imgs.to(device), labels.to(device)
			outputs = model(imgs)
			loss = criterion(outputs, labels)
			val_loss += loss.item()
			preds = outputs.argmax(dim=1)
			correct += (preds == labels).sum().item()
			total += labels.size(0)
	val_loss = val_loss / len(val_loader)
	val_accuracy = correct / total
	print(f"Epoch {epoch+1}/{epochs} |train loss:{train_loss:.4f} |val loss:{val_loss:.4f} |val accuracy:{val_accuracy:.4f}")

    # early stop
	if val_loss < best_val_loss:
		best_val_loss = val_loss
		counter = 0
		torch.save(model.state_dict(), "brain_tumor_v2.pth")
	else:
		counter += 1
		print(f"No improvement: {counter}/{patience}")
		if counter >= patience:
			print("Early stopping!!")
			break	

Epoch 1/30 |train loss:1.3481 |val loss:0.6318 |val accuracy:0.7297
Epoch 2/30 |train loss:0.5295 |val loss:0.5136 |val accuracy:0.7982
Epoch 3/30 |train loss:0.4443 |val loss:0.4245 |val accuracy:0.8342
Epoch 4/30 |train loss:0.4049 |val loss:0.4020 |val accuracy:0.8270
Epoch 5/30 |train loss:0.4120 |val loss:0.4238 |val accuracy:0.8342
No improvement: 1/5
Epoch 6/30 |train loss:0.3216 |val loss:0.3299 |val accuracy:0.8775
Epoch 7/30 |train loss:0.2967 |val loss:0.2991 |val accuracy:0.8703
Epoch 8/30 |train loss:0.2515 |val loss:0.2985 |val accuracy:0.8901
Epoch 9/30 |train loss:0.2975 |val loss:0.3420 |val accuracy:0.8577
No improvement: 1/5
Epoch 10/30 |train loss:0.2427 |val loss:0.3847 |val accuracy:0.8793
No improvement: 2/5
Epoch 11/30 |train loss:0.2748 |val loss:0.3010 |val accuracy:0.9009
No improvement: 3/5
Epoch 12/30 |train loss:0.1976 |val loss:0.2704 |val accuracy:0.9189
Epoch 13/30 |train loss:0.2075 |val loss:0.3083 |val accuracy:0.8973
No improvement: 1/5
Epoch 14/30 

In [16]:
model.eval() 
correct_test = 0
total_test = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

final_accuracy = (correct_test / total_test) * 100
print(f"\nFinal Test Accuracy: {final_accuracy:.2f}%")


Final Test Accuracy: 87.69%
